## 08c — Build TRAINING payloads (no upload)

**Objective (doc §3.2):** prepare tagged `TRAINING` JSON for TrustyAI.

### Why this notebook does **not** upload

Workbench → `trustyai-service:80` often **times out** (service is TLS / not plain HTTP on 80).
Validated lab path: **build here**, then upload from the laptop with `oc exec` into the TrustyAI pod (`curl http://127.0.0.1:8080/data/upload`).

### What this cell does

1. Infer via **`:8888`** (OVMS direct — no logger spam)
2. Write `/tmp/trustyai_training/train_*.json` (`data_tag: TRAINING`)

Keep small: **4** images (~2 MiB JSON each).

In [ ]:
import json
from pathlib import Path

import numpy as np
import requests
from PIL import Image

MODEL = "muffin-chihuahua"
PRED = "http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:8888"
TEST_ROOT = Path("/opt/app-root/src/data/muffin-chihuahua/test")
OUT = Path("/tmp/trustyai_training")
N_PER_CLASS = 2
SESSION = requests.Session()

OUT.mkdir(parents=True, exist_ok=True)


def preprocess(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype("float32") / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype="float32")
    std = np.array([0.229, 0.224, 0.225], dtype="float32")
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...].astype(np.float32)


def build_payload(path: Path) -> dict:
    chw = preprocess(path)
    request = {
        "inputs": [
            {
                "name": "input",
                "shape": list(chw.shape),
                "datatype": "FP32",
                "data": chw.ravel().tolist(),
            }
        ]
    }
    r = SESSION.post(
        f"{PRED}/v2/models/{MODEL}/infer",
        headers={"Content-Type": "application/json"},
        data=json.dumps(request),
        timeout=180,
    )
    r.raise_for_status()
    response = r.json()
    return {
        "model_name": MODEL,
        "data_tag": "TRAINING",
        "request": request,
        "response": {
            "model_name": response.get("model_name", MODEL),
            "model_version": response.get("model_version", "1"),
            "outputs": response["outputs"],
        },
    }


paths = []
paths += sorted((TEST_ROOT / "muffin").glob("*.jpg"))[:N_PER_CLASS]
paths += sorted((TEST_ROOT / "chihuahua").glob("*.jpg"))[:N_PER_CLASS]
if not paths:
    raise SystemExit(f"No images under {TEST_ROOT}")

print("Building", len(paths), "payloads in", OUT)
for i, p in enumerate(paths):
    print(f"[{i+1}/{len(paths)}] {p.name} infer :8888 …")
    payload = build_payload(p)
    out = OUT / f"train_{i}.json"
    out.write_text(json.dumps(payload))
    print(f"  wrote {out} ({out.stat().st_size} bytes)")

print("done — files:")
for f in sorted(OUT.glob("train_*.json")):
    print(" ", f, f.stat().st_size)

### Next (laptop) — upload via TrustyAI pod

Copy-paste on your machine (workbench pod is found automatically):

```bash
NS=chihuahua-vs-muffin-jan
WB=$(oc -n "$NS" get pods --field-selector=status.phase=Running -o name \
  | grep -E 'jupyter|workbench|notebook|chihuahua-muffin' | head -1 | cut -d/ -f2)
TAI=$(oc -n "$NS" get pod -l app=trustyai-service -o jsonpath='{.items[0].metadata.name}')
echo "WB=$WB  TAI=$TAI"

mkdir -p /tmp/trustyai_training
oc -n "$NS" cp "$WB:/tmp/trustyai_training/" /tmp/trustyai_training/

for f in /tmp/trustyai_training/train_*.json; do
  base=$(basename "$f")
  echo "=== upload $base ==="
  oc -n "$NS" cp "$f" "$TAI:/tmp/$base" -c trustyai-service
  oc -n "$NS" exec "$TAI" -c trustyai-service -- \
    curl -sS -X POST http://127.0.0.1:8080/data/upload \
      -H 'Content-Type: application/json' \
      --data-binary "@$base"
  # curl runs with cwd? use absolute path:
  # If the above fails, use: --data-binary @/tmp/$base
  echo
done
```

Prefer this safer loop (absolute path inside the pod):

```bash
NS=chihuahua-vs-muffin-jan
WB=$(oc -n "$NS" get pods --field-selector=status.phase=Running -o name \
  | grep -E 'jupyter|workbench|notebook|chihuahua-muffin' | head -1 | cut -d/ -f2)
TAI=$(oc -n "$NS" get pod -l app=trustyai-service -o jsonpath='{.items[0].metadata.name}')
echo "WB=$WB  TAI=$TAI"

rm -rf /tmp/trustyai_training && mkdir -p /tmp/trustyai_training
oc -n "$NS" cp "$WB:/tmp/trustyai_training/" /tmp/trustyai_training/
ls -la /tmp/trustyai_training/

for f in /tmp/trustyai_training/train_*.json; do
  base=$(basename "$f")
  echo "=== upload $base ==="
  oc -n "$NS" cp "$f" "$TAI:/tmp/$base" -c trustyai-service
  oc -n "$NS" exec "$TAI" -c trustyai-service -- \
    curl -sS -X POST http://127.0.0.1:8080/data/upload \
      -H 'Content-Type: application/json' \
      --data-binary "@/tmp/$base"
  echo
done

export TOKEN=$(oc whoami -t)
export TRUSTY_ROUTE=https://$(oc -n "$NS" get route/trustyai-service --template={{.spec.host}})
curl -sk -H "Authorization: Bearer $TOKEN" "$TRUSTY_ROUTE/info/tags"; echo
```
